# Can search reject a wrong answer as reliably as it confirms a correct one?

Search agents often guess an answer and then search to verify it, so the query contains the guess. This pilot tests, on the development questions of BrowseComp-Plus, whether such a query retrieves documents that merely mention the guess instead of evidence that supports or refutes it.

Outputs of this notebook are committed, so they show only aggregates, `query_id` and `docid`, never text from the dataset.

## 0. Setup

Paths and the seed used throughout, and a check that the downloaded files are in place. Row counts and column names come from the parquet metadata, so no row is read here.

In [1]:
import base64
import hashlib
import json
import re
from pathlib import Path

import bm25s
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

DATA = Path("data") if Path("data").exists() else Path("../data")
QUERIES_DIR = DATA / "browsecomp-plus" / "queries" / "data"
CORPUS_DIR = DATA / "browsecomp-plus" / "corpus" / "data"
RESULTS = DATA.parent / "results"
SEED = 0


def describe_parquet(directory: Path) -> None:
    files = sorted(directory.glob("*.parquet"))
    n_rows = sum(pq.ParquetFile(f).metadata.num_rows for f in files)
    columns = pq.ParquetFile(files[0]).schema_arrow.names
    print(f"{directory.parent.name}: {len(files)} files, {n_rows} rows, columns: {columns}")


describe_parquet(QUERIES_DIR)
describe_parquet(CORPUS_DIR)

queries: 6 files, 830 rows, columns: ['query_id', 'query', 'answer', 'gold_docs', 'negative_docs', 'evidence_docs']
corpus: 7 files, 100195 rows, columns: ['docid', 'text', 'url']


## 1. Development and test split

The questions are split at random before any result is computed, so that the choice of questions cannot depend on what we see. All prompts, query templates and thresholds are tuned on the development questions only; the test questions stay untouched until the final run. Only `query_id` is read here, and the split is saved to `results/split.json`.

In [2]:
N_DEV = 100
SPLIT_PATH = RESULTS / "split.json"

query_ids = sorted(
    query_id
    for f in QUERIES_DIR.glob("*.parquet")
    for query_id in pq.read_table(f, columns=["query_id"]).column("query_id").to_pylist()
)
shuffled = np.random.default_rng(SEED).permutation(query_ids).tolist()
split = {"seed": SEED, "dev": sorted(shuffled[:N_DEV]), "test": sorted(shuffled[N_DEV:])}

# The split must stay fixed once saved, so a mismatch stops the notebook instead of replacing it.
if SPLIT_PATH.exists():
    if json.loads(SPLIT_PATH.read_text()) != split:
        raise ValueError(f"{SPLIT_PATH} differs from the split computed here")
    print(f"{SPLIT_PATH} already exists and matches")
else:
    RESULTS.mkdir(exist_ok=True)
    SPLIT_PATH.write_text(json.dumps(split, indent=1) + "\n")
    print(f"Wrote {SPLIT_PATH}")

print(f"Questions: {len(query_ids)} total, {len(set(query_ids))} unique query_id")
print(f"Development: {len(split['dev'])}, test: {len(split['test'])}, "
      f"in both: {len(set(split['dev']) & set(split['test']))}")

../results/split.json already exists and matches
Questions: 830 total, 830 unique query_id
Development: 100, test: 730, in both: 0


## 2. De-obfuscating the development questions

Only the development questions are decrypted, to `data/browsecomp-plus/dev.jsonl`, which git ignores. Each field is base64-decoded and XORed with a repeated SHA-256 digest, as described on the dataset card. To check the result, every document is compared with the corpus. The counts below show how large each question's candidate pool is and whether the gold, evidence and negative lists overlap.

In [3]:
DEV_PATH = DATA / "browsecomp-plus" / "dev.jsonl"
CARD_PATH = QUERIES_DIR.parent / "README.md"
DOC_FIELDS = ["gold_docs", "evidence_docs", "negative_docs"]

# The key comes from the local dataset card so that it never appears in this notebook.
key_digest = hashlib.sha256(CARD_PATH.read_text(encoding="utf-8").strip().splitlines()[-1].encode("utf-8")).digest()


def decrypt_string(ciphertext: str) -> str:
    encrypted = np.frombuffer(base64.b64decode(ciphertext), dtype=np.uint8)
    key = np.resize(np.frombuffer(key_digest, dtype=np.uint8), len(encrypted))
    return (encrypted ^ key).tobytes().decode("utf-8")


def decrypt_row(row: dict) -> dict:
    decrypted = {"query_id": row["query_id"], "query": decrypt_string(row["query"]), "answer": decrypt_string(row["answer"])}
    for field in DOC_FIELDS:
        decrypted[field] = [{k: decrypt_string(v) for k, v in doc.items()} for doc in row[field]]
    return decrypted


dev_table = pq.read_table(QUERIES_DIR, filters=[("query_id", "in", split["dev"])])
dev_rows = sorted((decrypt_row(row) for row in dev_table.to_pylist()), key=lambda row: row["query_id"])

with DEV_PATH.open("w", encoding="utf-8") as f:
    for row in dev_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

dev_docs = [doc for row in dev_rows for field in DOC_FIELDS for doc in row[field]]
dev_docids = {doc["docid"] for doc in dev_docs}
corpus_table = pq.read_table(CORPUS_DIR, columns=["docid", "text"], filters=[("docid", "in", list(dev_docids))])
corpus_text = dict(zip(corpus_table.column("docid").to_pylist(), corpus_table.column("text").to_pylist()))

print(f"Decrypted {len(dev_rows)} development questions, {len(dev_docs)} document entries, {len(dev_docids)} unique docid")
print(f"Unique docid found in the corpus: {sum(d in corpus_text for d in dev_docids)} of {len(dev_docids)}")
print(f"Entries with text identical to the corpus: {sum(corpus_text.get(d['docid']) == d['text'] for d in dev_docs)} of {len(dev_docs)}")

doc_counts = pd.DataFrame([
    {
        "gold": len(gold := {d["docid"] for d in row["gold_docs"]}),
        "evidence": len(evidence := {d["docid"] for d in row["evidence_docs"]}),
        "negative": len(negative := {d["docid"] for d in row["negative_docs"]}),
        "pool": len(gold | evidence | negative),
        "gold_in_evidence": len(gold & evidence),
        "gold_in_negative": len(gold & negative),
        "evidence_in_negative": len(evidence & negative),
    }
    for row in dev_rows
])
print(f"Questions without a gold document: {(doc_counts['gold'] == 0).sum()}")
print(doc_counts.describe().loc[["mean", "min", "50%", "max"]].round(2).to_string())

Decrypted 100 development questions, 8475 document entries, 8173 unique docid
Unique docid found in the corpus: 8173 of 8173
Entries with text identical to the corpus: 8475 of 8475
Questions without a gold document: 0
      gold  evidence  negative    pool  gold_in_evidence  gold_in_negative  evidence_in_negative
mean  2.59      6.19     75.97   82.16              2.59               0.0                   0.0
min   1.00      2.00      1.00   12.00              1.00               0.0                   0.0
50%   2.00      6.00     73.50   79.00              2.00               0.0                   0.0
max   7.00     11.00    167.00  173.00              7.00               0.0                   0.0


## 3. BM25 baseline with the neutral query

BM25 is a classic keyword ranking: a document scores higher when it contains the query's rare words often, with a correction for document length. Here each development question gets its own index over its own candidate pool, and the question itself is the query (q0). Parameters are the `bm25s` defaults, with English stopwords and no stemming, and nothing is tuned.

Pool sizes differ a lot between questions, so each result is shown next to what a random ranking would give for the same question. The intervals are 95% bootstrap intervals over questions.

In [4]:
K_VALUES = [5, 10, 20]
N_BOOT = 2000


def build_pool(row: dict, rng: np.random.Generator) -> list[dict]:
    docs = {doc["docid"]: doc for field in DOC_FIELDS for doc in row[field]}
    # Gold documents are listed first, and BM25 score ties would otherwise favour them.
    return [docs[docid] for docid in rng.permutation(list(docs))]


def rank_pool(query: str, pool: list[dict]) -> list[str]:
    retriever = bm25s.BM25()
    retriever.index(bm25s.tokenize([doc["text"] for doc in pool], stopwords="en", show_progress=False), show_progress=False)
    query_tokens = bm25s.tokenize([query], stopwords="en", return_ids=False, show_progress=False)
    doc_indices, _ = retriever.retrieve(query_tokens, k=len(pool), show_progress=False)
    return [pool[i]["docid"] for i in doc_indices[0]]


def score_ranking(ranking: list[str], gold: set[str]) -> dict:
    n = len(ranking)
    scores = {"pool": n, "first_gold_rank": next(i for i, docid in enumerate(ranking, start=1) if docid in gold)}
    # Under a random ranking the expected first gold rank is (n + 1) / (g + 1).
    scores["random_first_gold_rank"] = (n + 1) / (len(gold) + 1)
    for k in K_VALUES:
        scores[f"recall@{k}"] = len(gold & set(ranking[:k])) / len(gold)
        scores[f"random_recall@{k}"] = min(k, n) / n
    return scores


def bootstrap_mean_ci(values: np.ndarray, n_boot: int = N_BOOT, level: float = 0.95, seed: int = SEED) -> tuple[float, float]:
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), (n_boot, len(values)))].mean(axis=1)
    tail = (1 - level) / 2 * 100
    return tuple(np.percentile(means, [tail, 100 - tail]))


pool_rng = np.random.default_rng(SEED)
q0_scores = pd.DataFrame([
    {"query_id": row["query_id"], **score_ranking(rank_pool(row["query"], build_pool(row, pool_rng)), {d["docid"] for d in row["gold_docs"]})}
    for row in dev_rows
])
q0_scores.to_csv(RESULTS / "bm25_q0_dev.csv", index=False)

summary = []
for metric in [f"recall@{k}" for k in K_VALUES] + ["first_gold_rank"]:
    difference = (q0_scores[metric] - q0_scores[f"random_{metric}"]).to_numpy()
    low, high = bootstrap_mean_ci(q0_scores[metric].to_numpy())
    diff_low, diff_high = bootstrap_mean_ci(difference)
    summary.append({
        "metric": metric, "bm25": q0_scores[metric].mean(), "ci_low": low, "ci_high": high,
        "random": q0_scores[f"random_{metric}"].mean(), "difference": difference.mean(), "diff_ci_low": diff_low, "diff_ci_high": diff_high,
    })

print(f"BM25 with q0 on {len(q0_scores)} development questions; pools of {q0_scores['pool'].min()}-{q0_scores['pool'].max()} documents")
print(f"Median first gold rank: {q0_scores['first_gold_rank'].median()}")
print(f"Pools of at most k documents: " + ", ".join(f"k={k}: {(q0_scores['pool'] <= k).sum()}" for k in K_VALUES))
print(pd.DataFrame(summary).round(3).to_string(index=False))

BM25 with q0 on 100 development questions; pools of 12-173 documents
Median first gold rank: 20.0
Pools of at most k documents: k=5: 0, k=10: 0, k=20: 1
         metric   bm25  ci_low  ci_high  random  difference  diff_ci_low  diff_ci_high
       recall@5  0.060   0.034    0.089   0.072      -0.012       -0.040         0.017
      recall@10  0.148   0.099    0.199   0.144       0.004       -0.041         0.049
      recall@20  0.277   0.215    0.346   0.282      -0.004       -0.063         0.059
first_gold_rank 28.570  24.069   33.261  27.165       1.405       -2.124         5.052


## 4. Is the baseline a pool effect?

Inside the pools, BM25 with q0 was not distinguishable from a random ranking. This section first checks that the ranking itself works, then runs the same query over the full corpus. The full corpus was added to the pilot after seeing the pool result, and the reason is recorded in `LOG.md`.

The index is built once and kept under `data/`. Documents with a score of zero have an arbitrary order, so gold documents with a zero score are counted separately.

In [5]:
def tokenize(texts: list[str]) -> list[list[str]]:
    return bm25s.tokenize(texts, stopwords="en", return_ids=False, show_progress=False)


def pool_diagnostics(row: dict, rng: np.random.Generator) -> dict:
    pool = build_pool(row, rng)
    gold = {d["docid"] for d in row["gold_docs"]}
    doc_tokens = tokenize([doc["text"] for doc in pool])
    query_tokens = tokenize([row["query"]])
    retriever = bm25s.BM25()
    retriever.index(doc_tokens, show_progress=False)
    doc_indices, doc_scores = retriever.retrieve(query_tokens, k=len(pool), show_progress=False)
    scores = np.empty(len(pool))
    scores[doc_indices[0]] = doc_scores[0]

    is_gold = np.array([doc["docid"] in gold for doc in pool])
    percentiles = [(scores < s).mean() + 0.5 * (scores == s).mean() for s in scores[is_gold]]
    query_terms = set(query_tokens[0])
    term_share = np.array([len(query_terms & set(tokens)) / len(query_terms) for tokens in doc_tokens])
    lengths = np.array([len(doc["text"]) for doc in pool])

    # Known-answer test: a passage copied from a document should rank that document first.
    target = int(rng.integers(len(pool)))
    words = pool[target]["text"].split()
    self_hit = np.nan
    if len(words) >= 200:
        probe = " ".join(words[len(words) // 2:len(words) // 2 + 60])
        self_hit = float(retriever.retrieve(tokenize([probe]), k=1, show_progress=False)[0][0][0] == target)

    return {
        "query_id": row["query_id"],
        "query_tokens": len(query_tokens[0]),
        "all_zero": bool((scores == 0).all()),
        "nonzero_share": (scores > 0).mean(),
        "gold_percentile": np.mean(percentiles),
        "self_hit": self_hit,
        "term_share_gold": term_share[is_gold].mean(),
        "term_share_other": term_share[~is_gold].mean(),
        "median_chars_gold": np.median(lengths[is_gold]),
        "median_chars_other": np.median(lengths[~is_gold]),
    }


diagnostics_rng = np.random.default_rng(SEED)
diagnostics = pd.DataFrame([pool_diagnostics(row, diagnostics_rng) for row in dev_rows])
diagnostics.to_csv(RESULTS / "bm25_pool_diagnostics_dev.csv", index=False)

low, high = bootstrap_mean_ci(diagnostics["gold_percentile"].to_numpy())
print("Inside the pools")
print(f"  Questions with all scores zero: {diagnostics['all_zero'].sum()}; mean share of pool with a score above zero: {diagnostics['nonzero_share'].mean():.3f}")
print(f"  Query tokens: mean {diagnostics['query_tokens'].mean():.1f}, min {diagnostics['query_tokens'].min()}")
print(f"  Mean gold score percentile (0.5 = chance): {diagnostics['gold_percentile'].mean():.3f}, 95% CI {low:.3f}-{high:.3f}")
print(f"  Copied passage ranks its document first: {int(diagnostics['self_hit'].sum())} of {diagnostics['self_hit'].notna().sum()}")
print(f"  Share of query terms in a document: gold {diagnostics['term_share_gold'].mean():.3f}, other {diagnostics['term_share_other'].mean():.3f}")
print(f"  Median document length in characters: gold {diagnostics['median_chars_gold'].median():.0f}, other {diagnostics['median_chars_other'].median():.0f}")

CORPUS_K_VALUES = [5, 10, 20, 100, 1000]
INDEX_PATH = DATA / "browsecomp-plus" / "bm25_corpus_index"
DOCIDS_PATH = DATA / "browsecomp-plus" / "bm25_corpus_docids.json"

# Building the index needs a lot of memory, so it runs separately: scripts/build_corpus_index.py.
if not DOCIDS_PATH.exists():
    raise FileNotFoundError(f"{DOCIDS_PATH} is missing; run scripts/build_corpus_index.py first")
corpus_retriever = bm25s.BM25.load(INDEX_PATH, load_corpus=False)
corpus_docids = json.loads(DOCIDS_PATH.read_text())
docid_position = {docid: i for i, docid in enumerate(corpus_docids)}
n_corpus = len(corpus_docids)


def corpus_scores(row: dict, query: str) -> dict:
    gold_positions = np.array(sorted({docid_position[d["docid"]] for d in row["gold_docs"]}))
    doc_indices, doc_scores = corpus_retriever.retrieve(tokenize([query]), k=n_corpus, show_progress=False)
    matched = np.isin(doc_indices[0], gold_positions)
    gold_ranks = np.flatnonzero(matched) + 1
    scores = {
        "query_id": row["query_id"],
        "n_gold": len(gold_positions),
        "first_gold_rank": gold_ranks.min(),
        "gold_zero_score": int((doc_scores[0][matched] == 0).sum()),
    }
    for k in CORPUS_K_VALUES:
        scores[f"recall@{k}"] = (gold_ranks <= k).mean()
    return scores


q0_corpus = pd.DataFrame([corpus_scores(row, row["query"]) for row in dev_rows])
q0_corpus.to_csv(RESULTS / "bm25_q0_corpus_dev.csv", index=False)

corpus_summary = []
for k in CORPUS_K_VALUES:
    low, high = bootstrap_mean_ci(q0_corpus[f"recall@{k}"].to_numpy())
    corpus_summary.append({"metric": f"recall@{k}", "bm25": q0_corpus[f"recall@{k}"].mean(), "ci_low": low, "ci_high": high, "random": k / n_corpus})

print(f"\nFull corpus: {n_corpus} documents, {len(q0_corpus)} development questions")
print(f"  Median first gold rank: {q0_corpus['first_gold_rank'].median()}; questions with a gold document in the top 10: {(q0_corpus['first_gold_rank'] <= 10).sum()}")
print(f"  Gold documents with score zero: {q0_corpus['gold_zero_score'].sum()} of {q0_corpus['n_gold'].sum()}")
print(pd.DataFrame(corpus_summary).round(5).to_string(index=False))

Inside the pools
  Questions with all scores zero: 0; mean share of pool with a score above zero: 0.983
  Query tokens: mean 62.7, min 26
  Mean gold score percentile (0.5 = chance): 0.494, 95% CI 0.458-0.531
  Copied passage ranks its document first: 87 of 88
  Share of query terms in a document: gold 0.341, other 0.376
  Median document length in characters: gold 5227, other 9776



Full corpus: 100195 documents, 100 development questions
  Median first gold rank: 1635.5; questions with a gold document in the top 10: 8
  Gold documents with score zero: 3 of 259
     metric    bm25  ci_low  ci_high  random
   recall@5 0.03000 0.00667  0.05833 0.00005
  recall@10 0.03667 0.01333  0.06667 0.00010
  recall@20 0.03917 0.01498  0.07000 0.00020
 recall@100 0.07750 0.04167  0.11667 0.00100
recall@1000 0.24950 0.18532  0.31867 0.00998


## 5. A neutral keyword query

BM25 already ignores stopwords, so the full question (q0) is already a set of keywords, on average more than 60 of them. A real keyword query has to choose. Here q_keywords is the 20 terms of the question that are rarest in the corpus, with no language model involved. The same terms will later be combined with each candidate, so that the difference between those queries and q_keywords comes from the candidate alone.

The value 20 was chosen before seeing any result for it and is not tuned. The values 10 and 40 only show how much the result depends on this choice. Differences are paired by question.

In [6]:
N_KEYWORDS = 20
N_KEYWORDS_SENSITIVITY = [10, 40]

# Each column of the score matrix holds the documents that contain one term.
doc_freq = np.diff(corpus_retriever.scores["indptr"])
vocab = corpus_retriever.vocab_dict


def select_keywords(query: str, n: int) -> str:
    terms = {term for term in tokenize([query])[0] if term in vocab}
    return " ".join(sorted(terms, key=lambda term: (doc_freq[vocab[term]], term))[:n])


# Check the document frequencies against a direct count of documents with a positive score.
check_terms = sorted({term for term in tokenize([dev_rows[0]["query"]])[0] if term in vocab})[:5]
_, check_scores = corpus_retriever.retrieve([[term] for term in check_terms], k=n_corpus, show_progress=False)
print(f"Document frequency matches a direct count for {sum(int((s > 0).sum()) == doc_freq[vocab[t]] for t, s in zip(check_terms, check_scores))} of {len(check_terms)} terms")

keyword_rng = np.random.default_rng(SEED)
pools = [build_pool(row, keyword_rng) for row in dev_rows]
pool_rows, corpus_rows = [], []
for n in [N_KEYWORDS] + N_KEYWORDS_SENSITIVITY:
    for row, pool in zip(dev_rows, pools):
        query = select_keywords(row["query"], n)
        gold = {d["docid"] for d in row["gold_docs"]}
        n_terms = len(query.split())
        pool_rows.append({"n_keywords": n, "query_id": row["query_id"], "n_terms": n_terms, **score_ranking(rank_pool(query, pool), gold)})
        corpus_rows.append({"n_keywords": n, "n_terms": n_terms, **corpus_scores(row, query)})
keywords_pool = pd.DataFrame(pool_rows)
keywords_corpus = pd.DataFrame(corpus_rows)
keywords_pool.to_csv(RESULTS / "bm25_keywords_pool_dev.csv", index=False)
keywords_corpus.to_csv(RESULTS / "bm25_keywords_corpus_dev.csv", index=False)


def compare_with_q0(keyword_scores: pd.DataFrame, q0: pd.DataFrame, metrics: list[str], space: str) -> list[dict]:
    comparison = []
    for n, group in keyword_scores.groupby("n_keywords"):
        paired = q0.merge(group, on="query_id", suffixes=("_q0", "_kw"))
        for metric in metrics:
            difference = (paired[f"{metric}_kw"] - paired[f"{metric}_q0"]).to_numpy()
            low, high = bootstrap_mean_ci(difference)
            comparison.append({"space": space, "n": n, "metric": metric, "q0": paired[f"{metric}_q0"].mean(), "keywords": paired[f"{metric}_kw"].mean(), "difference": difference.mean(), "ci_low": low, "ci_high": high})
    return comparison


comparison = (
    compare_with_q0(keywords_pool, q0_scores, [f"recall@{k}" for k in K_VALUES] + ["first_gold_rank"], "pool")
    + compare_with_q0(keywords_corpus, q0_corpus, [f"recall@{k}" for k in CORPUS_K_VALUES] + ["first_gold_rank"], "corpus")
)

terms_used = keywords_corpus.groupby("n_keywords")["n_terms"].mean().round(1).to_dict()
zero_gold = keywords_corpus.groupby("n_keywords")["gold_zero_score"].sum().to_dict()
median_rank = keywords_corpus.groupby("n_keywords")["first_gold_rank"].median().to_dict()
print(f"Mean terms per query: {terms_used}")
print(f"Gold documents with score zero in the corpus, of {q0_corpus['n_gold'].sum()}: {zero_gold} (q0: {q0_corpus['gold_zero_score'].sum()})")
print(f"Median first gold rank in the corpus: {median_rank} (q0: {q0_corpus['first_gold_rank'].median()})")
print(pd.DataFrame(comparison).sort_values(["space", "n"], kind="stable").round(4).to_string(index=False))

Document frequency matches a direct count for 5 of 5 terms


Mean terms per query: {10: 10.0, 20: 20.0, 40: 38.3}
Gold documents with score zero in the corpus, of 259: {10: 62, 20: 18, 40: 4} (q0: 3)
Median first gold rank in the corpus: {10: 293.5, 20: 408.0, 40: 1116.5} (q0: 1635.5)
 space  n          metric        q0  keywords  difference     ci_low    ci_high
corpus 10        recall@5    0.0300    0.0840      0.0540     0.0183     0.0962
corpus 10       recall@10    0.0367    0.0990      0.0623     0.0223     0.1067
corpus 10       recall@20    0.0392    0.1160      0.0768     0.0376     0.1228
corpus 10      recall@100    0.0775    0.2022      0.1247     0.0793     0.1743
corpus 10     recall@1000    0.2495    0.4298      0.1803     0.1078     0.2512
corpus 10 first_gold_rank 7229.3300 7402.7400    173.4100 -3068.4412  3987.9775
corpus 20        recall@5    0.0300    0.0787      0.0487     0.0170     0.0858
corpus 20       recall@10    0.0367    0.1007      0.0640     0.0290     0.1030
corpus 20       recall@20    0.0392    0.1123      0.07

## 6. Adding the correct answer

q_true is q_keywords plus the correct answer. It shows how much a correct guess helps retrieval, compared with the same keywords alone. The result is paired by question, in the pools and in the full corpus.

A document mentions the answer if the answer appears in its text as a string, ignoring case and spacing. Gold labels are different: annotators marked documents that contain the answer, possibly in another form. So this section also counts how many gold documents contain the answer as a string, and how many non-gold pool documents do. Non-gold pool documents that contain it are not necessarily wrong labels, because a string can match by chance, especially a short one.

In [7]:
MENTION_K = 10


def normalize(text: str) -> str:
    return " ".join(text.casefold().split())


def mentions(candidate: str, text: str) -> bool:
    return normalize(candidate) in normalize(text)


def top_corpus_docids(query: str, k: int = MENTION_K) -> list[str]:
    doc_indices, _ = corpus_retriever.retrieve(tokenize([query]), k=k, show_progress=False)
    return [corpus_docids[i] for i in doc_indices[0]]


def paired_differences(treatment: pd.DataFrame, baseline: pd.DataFrame, metrics: list[str], space: str, name: str = "q_true") -> list[dict]:
    paired = baseline.merge(treatment, on="query_id", suffixes=("_base", "_new"))
    comparison = []
    for metric in metrics:
        difference = (paired[f"{metric}_new"] - paired[f"{metric}_base"]).to_numpy()
        low, high = bootstrap_mean_ci(difference)
        comparison.append({"space": space, "metric": metric, "q_keywords": paired[f"{metric}_base"].mean(), name: paired[f"{metric}_new"].mean(), "difference": difference.mean(), "ci_low": low, "ci_high": high})
    return comparison


keyword_queries = [select_keywords(row["query"], N_KEYWORDS) for row in dev_rows]
true_queries = [f"{keywords} {row['answer']}" for keywords, row in zip(keyword_queries, dev_rows)]

# Texts of the top corpus documents for both queries, read once.
corpus_top = {
    "keywords": [top_corpus_docids(query) for query in keyword_queries],
    "true": [top_corpus_docids(query) for query in true_queries],
}
top_ids = sorted({docid for tops in corpus_top.values() for top in tops for docid in top})
top_table = pq.read_table(CORPUS_DIR, columns=["docid", "text"], filters=[("docid", "in", top_ids)])
top_text = dict(zip(top_table.column("docid").to_pylist(), top_table.column("text").to_pylist()))
del top_table

true_pool_rows, true_corpus_rows, pool_mention_rows, corpus_mention_rows, answer_rows = [], [], [], [], []
for i, (row, pool) in enumerate(zip(dev_rows, pools)):
    gold = {d["docid"] for d in row["gold_docs"]}
    answer = row["answer"]
    text_by_id = {doc["docid"]: doc["text"] for doc in pool}
    pool_mentions = {docid: mentions(answer, text) for docid, text in text_by_id.items()}

    answer_terms = [term for term in tokenize([answer])[0] if term in vocab]
    answer_rows.append({
        "query_id": row["query_id"],
        "has_usable_term": bool(answer_terms),
        "new_terms": len(set(answer_terms) - set(keyword_queries[i].split())),
        "gold_mentioning": sum(pool_mentions[d] for d in gold),
        "gold_total": len(gold),
        "other_mentioning": sum(mentioned for docid, mentioned in pool_mentions.items() if docid not in gold),
        "other_total": len(pool) - len(gold),
    })

    true_ranking = rank_pool(true_queries[i], pool)
    keyword_ranking = rank_pool(keyword_queries[i], pool)
    true_pool_rows.append({"query_id": row["query_id"], **score_ranking(true_ranking, gold)})
    true_corpus_rows.append(corpus_scores(row, true_queries[i]))
    pool_mention_rows.append({
        "query_id": row["query_id"],
        "mention_top10_keywords": sum(pool_mentions[d] for d in keyword_ranking[:MENTION_K]),
        "mention_top10_true": sum(pool_mentions[d] for d in true_ranking[:MENTION_K]),
        "mention_top10_random": sum(pool_mentions.values()) * min(MENTION_K, len(pool)) / len(pool),
    })
    corpus_mention_rows.append({
        "query_id": row["query_id"],
        "mention_top10_keywords": sum(mentions(answer, top_text[d]) for d in corpus_top["keywords"][i]),
        "mention_top10_true": sum(mentions(answer, top_text[d]) for d in corpus_top["true"][i]),
    })

true_pool = pd.DataFrame(true_pool_rows)
true_corpus = pd.DataFrame(true_corpus_rows)
pool_mention = pd.DataFrame(pool_mention_rows)
corpus_mention = pd.DataFrame(corpus_mention_rows)
answer_stats = pd.DataFrame(answer_rows)
true_pool.to_csv(RESULTS / "bm25_q_true_pool_dev.csv", index=False)
true_corpus.to_csv(RESULTS / "bm25_q_true_corpus_dev.csv", index=False)
pool_mention.to_csv(RESULTS / "mention_q_true_pool_dev.csv", index=False)
corpus_mention.to_csv(RESULTS / "mention_q_true_corpus_dev.csv", index=False)
answer_stats.to_csv(RESULTS / "answer_stats_dev.csv", index=False)

baseline_pool = keywords_pool[keywords_pool["n_keywords"] == N_KEYWORDS]
baseline_corpus = keywords_corpus[keywords_corpus["n_keywords"] == N_KEYWORDS]
comparison = (
    paired_differences(true_pool, baseline_pool, [f"recall@{k}" for k in K_VALUES] + ["first_gold_rank"], "pool")
    + paired_differences(true_corpus, baseline_corpus, [f"recall@{k}" for k in CORPUS_K_VALUES] + ["first_gold_rank"], "corpus")
)
for space, table in [("pool", pool_mention), ("corpus", corpus_mention)]:
    difference = (table["mention_top10_true"] - table["mention_top10_keywords"]).to_numpy()
    low, high = bootstrap_mean_ci(difference)
    comparison.append({"space": space, "metric": "mention_top10", "q_keywords": table["mention_top10_keywords"].mean(), "q_true": table["mention_top10_true"].mean(), "difference": difference.mean(), "ci_low": low, "ci_high": high})

print(f"Answers with at least one term BM25 can use: {answer_stats['has_usable_term'].sum()} of {len(answer_stats)}")
print(f"Answers adding a term that is not already among the keywords: {(answer_stats['new_terms'] > 0).sum()} of {len(answer_stats)}")
print(f"Gold documents containing the answer as a string: {answer_stats['gold_mentioning'].sum()} of {answer_stats['gold_total'].sum()}")
print(f"Questions where no gold document contains it: {(answer_stats['gold_mentioning'] == 0).sum()}")
print(f"Non-gold pool documents containing the answer as a string: {answer_stats['other_mentioning'].sum()} of {answer_stats['other_total'].sum()}; questions with at least one: {(answer_stats['other_mentioning'] > 0).sum()}")
print(f"Expected answer mentions in the top {MENTION_K} of a pool under a random ranking: {pool_mention['mention_top10_random'].mean():.3f}")
print(f"Median first gold rank, q_keywords to q_true: pool {baseline_pool['first_gold_rank'].median()} to {true_pool['first_gold_rank'].median()}, corpus {baseline_corpus['first_gold_rank'].median()} to {true_corpus['first_gold_rank'].median()}")
print(f"Gold documents with score zero in the corpus, of {true_corpus['n_gold'].sum()}: q_keywords {baseline_corpus['gold_zero_score'].sum()}, q_true {true_corpus['gold_zero_score'].sum()}")
print(pd.DataFrame(comparison).sort_values("space", kind="stable").round(4).to_string(index=False))

Answers with at least one term BM25 can use: 100 of 100
Answers adding a term that is not already among the keywords: 100 of 100
Gold documents containing the answer as a string: 224 of 259
Questions where no gold document contains it: 13
Non-gold pool documents containing the answer as a string: 131 of 7957; questions with at least one: 15
Expected answer mentions in the top 10 of a pool under a random ranking: 0.486
Median first gold rank, q_keywords to q_true: pool 11.0 to 2.0, corpus 408.0 to 4.5
Gold documents with score zero in the corpus, of 259: q_keywords 18, q_true 1
 space          metric  q_keywords   q_true  difference     ci_low    ci_high
corpus        recall@5      0.0787   0.3620      0.2834     0.2164     0.3577
corpus       recall@10      0.1007   0.4031      0.3024     0.2319     0.3812
corpus       recall@20      0.1123   0.4396      0.3272     0.2513     0.4090
corpus      recall@100      0.1593   0.6057      0.4464     0.3616     0.5301
corpus     recall@1000    

## 7. A random wrong candidate

q_random is q_keywords plus the answer of another development question of the same coarse type. It is a control: the candidate has nothing to do with the question, so it shows what adding any wrong answer does to retrieval. It does not test H1, which needs a plausible wrong answer.

The type comes from a simple rule (date, number or other), fixed before seeing any result for this query. A candidate is rejected if it overlaps with the correct answer as a string, or if it appears in a gold document of the question. This is a string check, not a verification by a person. Questions left with no candidate are dropped and counted.

The candidates come from the same corpus: each is the answer to another question, whose gold documents are in the index. So the section also counts how many top documents are gold documents of the question the candidate came from, because a real wrong guess would not always have such documents. Candidate texts are stored only under `data/`. Each question gets one draw, so the result also carries the noise of that draw.

In [8]:
MONTHS = r"jan(uary)?|feb(ruary)?|mar(ch)?|apr(il)?|may|june?|july?|aug(ust)?|sep(t|tember)?|oct(ober)?|nov(ember)?|dec(ember)?"


def answer_type(answer: str) -> str:
    text = normalize(answer)
    if re.fullmatch(r"\d{4}", text) or (re.search(r"\d", text) and re.search(rf"\b({MONTHS})\b", text)):
        return "date"
    if re.match(r"\W?\d", text):
        return "number"
    return "other"


def pick_random_source(i: int, indices_by_type: dict, rng: np.random.Generator) -> int | None:
    row = dev_rows[i]
    answer = normalize(row["answer"])
    gold_texts = [normalize(doc["text"]) for doc in row["gold_docs"]]
    same_type = indices_by_type[answer_type(row["answer"])]
    for position in rng.permutation(len(same_type)):
        source = same_type[position]
        candidate = normalize(dev_rows[source]["answer"])
        if candidate in answer or answer in candidate or any(candidate in text for text in gold_texts):
            continue
        return source
    return None


def docs_with_all_terms(text: str) -> int:
    terms = set(tokenize([text])[0])
    if not terms or any(term not in vocab for term in terms):
        return 0
    indptr, indices = corpus_retriever.scores["indptr"], corpus_retriever.scores["indices"]
    shared = None
    for term in terms:
        postings = indices[indptr[vocab[term]]:indptr[vocab[term] + 1]]
        shared = postings if shared is None else np.intersect1d(shared, postings)
    return len(shared)


indices_by_type = {}
for i, row in enumerate(dev_rows):
    indices_by_type.setdefault(answer_type(row["answer"]), []).append(i)

random_rng = np.random.default_rng(SEED)
random_sources = {i: pick_random_source(i, indices_by_type, random_rng) for i in range(len(dev_rows))}
selected = [i for i, source in random_sources.items() if source is not None]
random_candidates = {i: dev_rows[random_sources[i]]["answer"] for i in selected}
random_queries = {i: f"{keyword_queries[i]} {random_candidates[i]}" for i in selected}

with open(DATA / "browsecomp-plus" / "q_random_dev.jsonl", "w") as f:
    for i in selected:
        record = {
            "query_id": dev_rows[i]["query_id"],
            "candidate": random_candidates[i],
            "source_query_id": dev_rows[random_sources[i]]["query_id"],
        }
        f.write(json.dumps(record) + "\n")

corpus_top["random"] = {i: top_corpus_docids(random_queries[i]) for i in selected}
missing_ids = sorted({docid for top in corpus_top["random"].values() for docid in top} - set(top_text))
missing_table = pq.read_table(CORPUS_DIR, columns=["docid", "text"], filters=[("docid", "in", missing_ids)])
top_text.update(zip(missing_table.column("docid").to_pylist(), missing_table.column("text").to_pylist()))
del missing_table

random_pool_rows, random_corpus_rows, random_pool_mention, random_corpus_mention, candidate_rows = [], [], [], [], []
for i in selected:
    row, pool, candidate = dev_rows[i], pools[i], random_candidates[i]
    query_id = row["query_id"]
    gold = {doc["docid"] for doc in row["gold_docs"]}
    source_gold = {doc["docid"] for doc in dev_rows[random_sources[i]]["gold_docs"]}
    pool_mentions = {doc["docid"]: mentions(candidate, doc["text"]) for doc in pool}

    candidate_terms = [term for term in tokenize([candidate])[0] if term in vocab]
    candidate_rows.append({
        "query_id": query_id,
        "new_terms": len(set(candidate_terms) - set(keyword_queries[i].split())),
        "pool_docs_mentioning": sum(pool_mentions.values()),
        "corpus_docs_with_candidate_terms": docs_with_all_terms(candidate),
        "corpus_docs_with_answer_terms": docs_with_all_terms(row["answer"]),
    })

    keyword_ranking = rank_pool(keyword_queries[i], pool)
    random_ranking = rank_pool(random_queries[i], pool)
    random_pool_rows.append({"query_id": query_id, **score_ranking(random_ranking, gold)})
    random_corpus_rows.append(corpus_scores(row, random_queries[i]))
    random_pool_mention.append({
        "query_id": query_id,
        "mention_top10_keywords": sum(pool_mentions[d] for d in keyword_ranking[:MENTION_K]),
        "mention_top10_q_random": sum(pool_mentions[d] for d in random_ranking[:MENTION_K]),
        "mention_top10_chance": sum(pool_mentions.values()) * min(MENTION_K, len(pool)) / len(pool),
        "top10_kept": len(set(keyword_ranking[:MENTION_K]) & set(random_ranking[:MENTION_K])),
    })
    random_top = corpus_top["random"][i]
    random_corpus_mention.append({
        "query_id": query_id,
        "mention_top10_keywords": sum(mentions(candidate, top_text[d]) for d in corpus_top["keywords"][i]),
        "mention_top10_q_random": sum(mentions(candidate, top_text[d]) for d in random_top),
        "source_gold_top10": len(source_gold & set(random_top)),
        "top10_kept": len(set(corpus_top["keywords"][i]) & set(random_top)),
    })

random_pool = pd.DataFrame(random_pool_rows)
random_corpus = pd.DataFrame(random_corpus_rows)
random_pool_mentions = pd.DataFrame(random_pool_mention)
random_corpus_mentions = pd.DataFrame(random_corpus_mention)
candidate_stats = pd.DataFrame(candidate_rows)
random_pool.to_csv(RESULTS / "bm25_q_random_pool_dev.csv", index=False)
random_corpus.to_csv(RESULTS / "bm25_q_random_corpus_dev.csv", index=False)
random_pool_mentions.to_csv(RESULTS / "mention_q_random_pool_dev.csv", index=False)
random_corpus_mentions.to_csv(RESULTS / "mention_q_random_corpus_dev.csv", index=False)
candidate_stats.to_csv(RESULTS / "candidate_stats_q_random_dev.csv", index=False)

selected_ids = set(candidate_stats["query_id"])
selected_base_pool = baseline_pool[baseline_pool["query_id"].isin(selected_ids)]
selected_base_corpus = baseline_corpus[baseline_corpus["query_id"].isin(selected_ids)]
comparison = (
    paired_differences(random_pool, selected_base_pool, [f"recall@{k}" for k in K_VALUES] + ["first_gold_rank"], "pool", "q_random")
    + paired_differences(random_corpus, selected_base_corpus, [f"recall@{k}" for k in CORPUS_K_VALUES] + ["first_gold_rank"], "corpus", "q_random")
)
for space, table in [("pool", random_pool_mentions), ("corpus", random_corpus_mentions)]:
    difference = (table["mention_top10_q_random"] - table["mention_top10_keywords"]).to_numpy()
    low, high = bootstrap_mean_ci(difference)
    comparison.append({"space": space, "metric": "mention_top10", "q_keywords": table["mention_top10_keywords"].mean(), "q_random": table["mention_top10_q_random"].mean(), "difference": difference.mean(), "ci_low": low, "ci_high": high})

source_low, source_high = bootstrap_mean_ci(random_corpus_mentions["source_gold_top10"].to_numpy(dtype=float))
print(f"Answer types: { {name: len(indices) for name, indices in sorted(indices_by_type.items())} }")
print(f"Questions with a candidate: {len(selected)} of {len(dev_rows)}")
print(f"Candidates under 4 characters: {sum(len(normalize(c)) < 4 for c in random_candidates.values())}")
print(f"Candidates adding a term that is not already among the keywords: {(candidate_stats['new_terms'] > 0).sum()} of {len(selected)}")
print(f"Questions with a pool document containing the candidate as a string: {(candidate_stats['pool_docs_mentioning'] > 0).sum()} of {len(selected)}")
print(f"Corpus documents with all terms, median: candidate {candidate_stats['corpus_docs_with_candidate_terms'].median()}, correct answer {candidate_stats['corpus_docs_with_answer_terms'].median()}")
print(f"Terms found together in no corpus document: candidate {(candidate_stats['corpus_docs_with_candidate_terms'] == 0).sum()}, correct answer {(candidate_stats['corpus_docs_with_answer_terms'] == 0).sum()}")
print(f"Expected candidate mentions in the top {MENTION_K} of a pool under a random ranking: {random_pool_mentions['mention_top10_chance'].mean():.3f}")
print(f"Mean top {MENTION_K} documents kept from q_keywords: pool {random_pool_mentions['top10_kept'].mean():.2f}, corpus {random_corpus_mentions['top10_kept'].mean():.2f}")
print(f"Gold documents of the candidate's source question in the corpus top {MENTION_K}: mean {random_corpus_mentions['source_gold_top10'].mean():.2f}, 95% CI {source_low:.2f}-{source_high:.2f}")
print(f"Median first gold rank, q_keywords to q_random: pool {selected_base_pool['first_gold_rank'].median()} to {random_pool['first_gold_rank'].median()}, corpus {selected_base_corpus['first_gold_rank'].median()} to {random_corpus['first_gold_rank'].median()}")
print(f"Gold documents with score zero in the corpus, of {random_corpus['n_gold'].sum()}: q_keywords {selected_base_corpus['gold_zero_score'].sum()}, q_random {random_corpus['gold_zero_score'].sum()}")
print(pd.DataFrame(comparison).sort_values("space", kind="stable").round(4).to_string(index=False))

Answer types: {'date': 7, 'number': 6, 'other': 87}
Questions with a candidate: 100 of 100
Candidates under 4 characters: 1
Candidates adding a term that is not already among the keywords: 100 of 100
Questions with a pool document containing the candidate as a string: 3 of 100
Corpus documents with all terms, median: candidate 74.5, correct answer 63.5
Terms found together in no corpus document: candidate 0, correct answer 0
Expected candidate mentions in the top 10 of a pool under a random ranking: 0.068
Mean top 10 documents kept from q_keywords: pool 9.08, corpus 8.02
Gold documents of the candidate's source question in the corpus top 10: mean 0.41, 95% CI 0.21-0.63
Median first gold rank, q_keywords to q_random: pool 11.0 to 12.5, corpus 408.0 to 451.0
Gold documents with score zero in the corpus, of 259: q_keywords 18, q_random 16
 space          metric  q_keywords  q_random  difference   ci_low  ci_high
corpus        recall@5      0.0787    0.0762     -0.0025  -0.0075   0.0000
co

## 8. A plausible wrong candidate

q_wrong is q_keywords plus a wrong answer that a model would actually guess. A small local model, Qwen3-4B (4-bit, on CPU, temperature 0, no reasoning), answers each question without search and gives three guesses. The first guess that passes the same string check as q_random (no overlap with the correct answer, not in a gold document) becomes the candidate. A first guess that matches the correct answer is counted as a correct closed-book guess. The prompt and the rule were fixed in `LOG.md` before any result.

This is the comparison for H1: if adding a plausible wrong candidate lowers gold recall relative to q_keywords, search is pulled toward the guess. It is placed next to q_true for H2: the gain from a correct guess against the loss from a wrong one. The model runs through `scripts/run_llm.py`, which resumes an unfinished run; candidates and raw outputs stay under `data/`.

In [9]:
import subprocess
import sys

MODEL_PATH = DATA / "models" / "Qwen3-4B-Q4_K_M.gguf"
LLM_DIR = DATA / "browsecomp-plus" / "llm"
RUN_LLM = True
GUESS_PROMPT = (
    "Answer the question below without searching. Give your three best guesses, most likely first, "
    "one per line, each a short answer only (a name, number, date or short phrase), with no explanation.\n\n"
    "Question: {question} /no_think"
)


def run_llm(name: str, prompts: list[dict], max_tokens: int) -> dict[str, str]:
    LLM_DIR.mkdir(exist_ok=True)
    prompts_path, outputs_path = LLM_DIR / f"{name}_prompts.jsonl", LLM_DIR / f"{name}_outputs.jsonl"
    prompts_path.write_text("".join(json.dumps(p, ensure_ascii=False) + "\n" for p in prompts), encoding="utf-8")
    if RUN_LLM:
        script = DATA.parent / "scripts" / "run_llm.py"
        command = [sys.executable, str(script), str(MODEL_PATH), str(prompts_path), str(outputs_path), "--max-tokens", str(max_tokens)]
        # Only the summary line is printed; the script's progress log would show nothing new here.
        log = subprocess.run(command, check=True, capture_output=True, text=True).stdout.splitlines()
        print(f"{name}: {log[0]}")
    outputs = {}
    if outputs_path.exists():
        outputs = {r["id"]: r["output"] for r in map(json.loads, outputs_path.read_text(encoding="utf-8").splitlines())}
    missing = {p["id"] for p in prompts} - set(outputs)
    if missing:
        raise RuntimeError(f"{name}: {len(missing)} prompts have no output; run with RUN_LLM = True")
    return outputs


def parse_guesses(output: str) -> list[str]:
    lines = [re.sub(r"^\s*(\d+[.)]|[-*•])\s*", "", line).strip().strip("\"'").rstrip(".").strip() for line in output.splitlines()]
    return [line for line in lines if line][:3]


def is_wrong_candidate(candidate: str, row: dict) -> bool:
    candidate, answer = normalize(candidate), normalize(row["answer"])
    if not candidate or candidate in answer or answer in candidate:
        return False
    return not any(candidate in normalize(doc["text"]) for doc in row["gold_docs"])


guess_prompts = [{"id": row["query_id"], "prompt": GUESS_PROMPT.format(question=row["query"])} for row in dev_rows]
guess_outputs = run_llm("guesses_dev", guess_prompts, max_tokens=64)

guess_rows, wrong_candidates = [], {}
for i, row in enumerate(dev_rows):
    guesses = parse_guesses(guess_outputs[row["query_id"]])
    answer = normalize(row["answer"])
    first = normalize(guesses[0]) if guesses else ""
    first_correct = bool(first) and (first in answer or answer in first)
    position = next((p for p, guess in enumerate(guesses) if is_wrong_candidate(guess, row)), None)
    guess_rows.append({"query_id": row["query_id"], "n_guesses": len(guesses), "first_correct": first_correct, "candidate_position": position})
    if position is not None:
        wrong_candidates[i] = guesses[position]

with open(DATA / "browsecomp-plus" / "q_wrong_dev.jsonl", "w", encoding="utf-8") as f:
    for i, candidate in wrong_candidates.items():
        f.write(json.dumps({"query_id": dev_rows[i]["query_id"], "candidate": candidate}, ensure_ascii=False) + "\n")

wrong_selected = sorted(wrong_candidates)
wrong_queries = {i: f"{keyword_queries[i]} {wrong_candidates[i]}" for i in wrong_selected}
corpus_top["wrong"] = {i: top_corpus_docids(wrong_queries[i]) for i in wrong_selected}
missing_ids = sorted({docid for top in corpus_top["wrong"].values() for docid in top} - set(top_text))
missing_table = pq.read_table(CORPUS_DIR, columns=["docid", "text"], filters=[("docid", "in", missing_ids)])
top_text.update(zip(missing_table.column("docid").to_pylist(), missing_table.column("text").to_pylist()))
del missing_table

wrong_pool_rows, wrong_corpus_rows, wrong_pool_mention, wrong_corpus_mention = [], [], [], []
for i in wrong_selected:
    row, pool, candidate = dev_rows[i], pools[i], wrong_candidates[i]
    query_id = row["query_id"]
    gold = {doc["docid"] for doc in row["gold_docs"]}
    pool_mentions = {doc["docid"]: mentions(candidate, doc["text"]) for doc in pool}
    candidate_terms = {term for term in tokenize([candidate])[0] if term in vocab}
    guess_rows[i].update({
        "new_terms": len(candidate_terms - set(keyword_queries[i].split())),
        "pool_docs_mentioning": sum(pool_mentions.values()),
        "corpus_docs_with_candidate_terms": docs_with_all_terms(candidate),
    })

    keyword_ranking = rank_pool(keyword_queries[i], pool)
    wrong_ranking = rank_pool(wrong_queries[i], pool)
    wrong_pool_rows.append({"query_id": query_id, **score_ranking(wrong_ranking, gold)})
    wrong_corpus_rows.append(corpus_scores(row, wrong_queries[i]))
    wrong_pool_mention.append({
        "query_id": query_id,
        "mention_top10_keywords": sum(pool_mentions[d] for d in keyword_ranking[:MENTION_K]),
        "mention_top10_q_wrong": sum(pool_mentions[d] for d in wrong_ranking[:MENTION_K]),
        "top10_kept": len(set(keyword_ranking[:MENTION_K]) & set(wrong_ranking[:MENTION_K])),
    })
    wrong_top = corpus_top["wrong"][i]
    wrong_corpus_mention.append({
        "query_id": query_id,
        "mention_top10_keywords": sum(mentions(candidate, top_text[d]) for d in corpus_top["keywords"][i]),
        "mention_top10_q_wrong": sum(mentions(candidate, top_text[d]) for d in wrong_top),
        "top10_kept": len(set(corpus_top["keywords"][i]) & set(wrong_top)),
    })

wrong_pool = pd.DataFrame(wrong_pool_rows)
wrong_corpus = pd.DataFrame(wrong_corpus_rows)
wrong_pool_mentions = pd.DataFrame(wrong_pool_mention)
wrong_corpus_mentions = pd.DataFrame(wrong_corpus_mention)
guess_stats = pd.DataFrame(guess_rows)
wrong_pool.to_csv(RESULTS / "bm25_q_wrong_pool_dev.csv", index=False)
wrong_corpus.to_csv(RESULTS / "bm25_q_wrong_corpus_dev.csv", index=False)
wrong_pool_mentions.to_csv(RESULTS / "mention_q_wrong_pool_dev.csv", index=False)
wrong_corpus_mentions.to_csv(RESULTS / "mention_q_wrong_corpus_dev.csv", index=False)
guess_stats.to_csv(RESULTS / "candidate_stats_q_wrong_dev.csv", index=False)

wrong_ids = set(wrong_pool["query_id"])
comparison = []
for space, metrics, base, wrong in [
    ("pool", [f"recall@{k}" for k in K_VALUES] + ["first_gold_rank"], baseline_pool, wrong_pool),
    ("corpus", [f"recall@{k}" for k in CORPUS_K_VALUES] + ["first_gold_rank"], baseline_corpus, wrong_corpus),
]:
    comparison += paired_differences(wrong, base[base["query_id"].isin(wrong_ids)], metrics, space, "q_wrong")
for space, table in [("pool", wrong_pool_mentions), ("corpus", wrong_corpus_mentions)]:
    difference = (table["mention_top10_q_wrong"] - table["mention_top10_keywords"]).to_numpy()
    low, high = bootstrap_mean_ci(difference)
    comparison.append({"space": space, "metric": "mention_top10", "q_keywords": table["mention_top10_keywords"].mean(), "q_wrong": table["mention_top10_q_wrong"].mean(), "difference": difference.mean(), "ci_low": low, "ci_high": high})

# H2 on retrieval: the gain in gold recall from the correct answer against the change from the wrong one, same questions.
asymmetry = []
for space, metrics, base, true, wrong in [
    ("pool", [f"recall@{k}" for k in K_VALUES], baseline_pool, true_pool, wrong_pool),
    ("corpus", [f"recall@{k}" for k in CORPUS_K_VALUES], baseline_corpus, true_corpus, wrong_corpus),
]:
    paired = base.merge(true, on="query_id", suffixes=("", "_true")).merge(wrong, on="query_id", suffixes=("", "_wrong"))
    for metric in metrics:
        gain = (paired[f"{metric}_true"] - paired[metric]).to_numpy()
        change = (paired[f"{metric}_wrong"] - paired[metric]).to_numpy()
        gain_low, gain_high = bootstrap_mean_ci(gain)
        change_low, change_high = bootstrap_mean_ci(change)
        asymmetry.append({"space": space, "metric": metric, "true_gain": gain.mean(), "true_ci": f"{gain_low:.3f} to {gain_high:.3f}", "wrong_change": change.mean(), "wrong_ci": f"{change_low:.3f} to {change_high:.3f}"})
pd.DataFrame(asymmetry).to_csv(RESULTS / "asymmetry_true_wrong_dev.csv", index=False)

print(f"Questions with three parsed guesses: {(guess_stats['n_guesses'] == 3).sum()} of {len(guess_stats)}")
print(f"First guess matches the correct answer (closed-book, string match): {guess_stats['first_correct'].sum()} of {len(guess_stats)}")
print(f"Questions with a wrong candidate: {len(wrong_selected)} of {len(dev_rows)}; position of the candidate among the guesses: {guess_stats['candidate_position'].value_counts().sort_index().to_dict()}")
print(f"Candidates adding a term that is not already among the keywords: {(guess_stats['new_terms'] > 0).sum()} of {len(wrong_selected)}")
print(f"Candidates whose terms occur together in no corpus document: {(guess_stats['corpus_docs_with_candidate_terms'] == 0).sum()}")
print(f"Questions with a pool document containing the candidate as a string: {(guess_stats['pool_docs_mentioning'] > 0).sum()} of {len(wrong_selected)}")
print(f"Mean top {MENTION_K} documents kept from q_keywords: pool {wrong_pool_mentions['top10_kept'].mean():.2f}, corpus {wrong_corpus_mentions['top10_kept'].mean():.2f}")
print(f"Median first gold rank, q_keywords to q_wrong: corpus {baseline_corpus[baseline_corpus['query_id'].isin(wrong_ids)]['first_gold_rank'].median()} to {wrong_corpus['first_gold_rank'].median()}")
print(pd.DataFrame(comparison).sort_values("space", kind="stable").round(4).to_string(index=False))
print("\nChange in gold recall relative to q_keywords, on the same questions (H2)")
print(pd.DataFrame(asymmetry).round(4).to_string(index=False))

guesses_dev: 100 prompts, 100 already done, 0 to run


Questions with three parsed guesses: 88 of 100
First guess matches the correct answer (closed-book, string match): 2 of 100
Questions with a wrong candidate: 97 of 100; position of the candidate among the guesses: {0.0: 91, 1.0: 5, 2.0: 1}
Candidates adding a term that is not already among the keywords: 93 of 97
Candidates whose terms occur together in no corpus document: 4
Questions with a pool document containing the candidate as a string: 37 of 97
Mean top 10 documents kept from q_keywords: pool 8.69, corpus 7.10
Median first gold rank, q_keywords to q_wrong: corpus 473.0 to 394.0
 space          metric  q_keywords   q_wrong  difference    ci_low  ci_high
corpus        recall@5      0.0759    0.0682     -0.0077   -0.0206   0.0000
corpus       recall@10      0.0986    0.0837     -0.0149   -0.0304  -0.0034
corpus       recall@20      0.1107    0.1029     -0.0077   -0.0172   0.0000
corpus      recall@100      0.1591    0.1631      0.0040   -0.0084   0.0189
corpus     recall@1000      0

## 9. Does the reader accept the wrong candidate? (H4)

Retrieval alone does not show whether the echo misleads anyone. Here the same model reads the top 5 corpus results of a query and judges a proposed answer as SUPPORTED, REFUTED or UNCLEAR. Each document is cut to the 200 words that contain the most query terms, like a search snippet, because whole documents do not fit the model's speed on a CPU.

There are four conditions per question: the wrong candidate judged on the results of q_keywords and of q_wrong, and the correct answer judged on the results of q_keywords and of q_true. The main H4 number is the paired difference in how often the wrong candidate is called SUPPORTED: q_wrong results minus q_keywords results. Whether a gold document is in the top 5 serves as a rough automatic stand-in for refuting evidence, until the manual labels exist. Verdicts are saved in `results/`; prompts and raw outputs stay under `data/`.

In [10]:
READER_K = 5
SNIPPET_WORDS = 200
SNIPPET_STEP = 50
VERDICTS = ["SUPPORTED", "REFUTED", "UNCLEAR"]
READER_PROMPT = (
    "You are checking a proposed answer to a question using search results.\n\n"
    "Question: {question}\n\nSearch results:\n{results}\n\nProposed answer: {candidate}\n\n"
    "Based only on the search results, is the proposed answer correct? Reply with exactly one word: "
    "SUPPORTED if the results show it is correct, REFUTED if they show it is wrong, "
    "or UNCLEAR if they do not say. /no_think"
)


def snippet(text: str, query: str) -> str:
    # Same term pattern as the bm25s tokenizer, so the window follows what the ranking matched.
    terms = set(tokenize([query])[0])
    words = text.split()
    hits = np.array([any(t in terms for t in re.findall(r"\b\w\w+\b", word.lower())) for word in words], dtype=int)
    if len(words) <= SNIPPET_WORDS:
        return " ".join(words)
    starts = range(0, len(words) - SNIPPET_WORDS + 1, SNIPPET_STEP)
    best = max(starts, key=lambda s: (hits[s:s + SNIPPET_WORDS].sum(), -s))
    return " ".join(words[best:best + SNIPPET_WORDS])


def results_block(docids: list[str], query: str) -> str:
    return "\n\n".join(f"[{n}] {snippet(top_text[d], query)}" for n, d in enumerate(docids[:READER_K], start=1))


def parse_verdict(output: str) -> str:
    first = re.findall(r"[A-Za-z]+", output)
    return first[0].upper() if first and first[0].upper() in VERDICTS else "OTHER"


# Prompts that share the same results are adjacent, so the model can reuse the shared prefix.
reader_prompts, reader_meta = [], []
for i in wrong_selected:
    row = dev_rows[i]
    keyword_results = results_block(corpus_top["keywords"][i], keyword_queries[i])
    conditions = [
        ("wrong_keywords", wrong_candidates[i], keyword_results, corpus_top["keywords"][i]),
        ("true_keywords", row["answer"], keyword_results, corpus_top["keywords"][i]),
        ("wrong_wrong", wrong_candidates[i], results_block(corpus_top["wrong"][i], wrong_queries[i]), corpus_top["wrong"][i]),
        ("true_true", row["answer"], results_block(corpus_top["true"][i], true_queries[i]), corpus_top["true"][i]),
    ]
    gold = {doc["docid"] for doc in row["gold_docs"]}
    for condition, candidate, results, docids in conditions:
        prompt = READER_PROMPT.format(question=row["query"], results=results, candidate=candidate)
        reader_prompts.append({"id": f"{row['query_id']}|{condition}", "prompt": prompt})
        reader_meta.append({
            "query_id": row["query_id"], "condition": condition,
            "gold_in_top5": len(gold & set(docids[:READER_K])),
            "candidate_in_top5": sum(mentions(candidate, top_text[d]) for d in docids[:READER_K]),
        })

reader_outputs = run_llm("reader_dev", reader_prompts, max_tokens=16)
reader = pd.DataFrame(reader_meta)
reader["verdict"] = [parse_verdict(reader_outputs[p["id"]]) for p in reader_prompts]
reader.to_csv(RESULTS / "reader_dev.csv", index=False)

wide = reader.pivot(index="query_id", columns="condition", values="verdict")
gold_wide = reader.pivot(index="query_id", columns="condition", values="gold_in_top5")
shares = reader.groupby("condition")["verdict"].value_counts(normalize=True).unstack(fill_value=0)
shares["n"] = reader.groupby("condition").size()
shares["gold_in_top5"] = reader.groupby("condition")["gold_in_top5"].apply(lambda s: (s > 0).mean())
shares["candidate_in_top5"] = reader.groupby("condition")["candidate_in_top5"].mean()

reader_summary = []
for name, treated, base in [("H4 wrong candidate", "wrong_wrong", "wrong_keywords"), ("correct answer", "true_true", "true_keywords")]:
    for verdict in ["SUPPORTED", "REFUTED"]:
        difference = ((wide[treated] == verdict).astype(float) - (wide[base] == verdict).astype(float)).to_numpy()
        low, high = bootstrap_mean_ci(difference)
        reader_summary.append({"comparison": f"{name}: {treated} - {base}", "verdict": verdict, "base": (wide[base] == verdict).mean(), "treated": (wide[treated] == verdict).mean(), "difference": difference.mean(), "ci_low": low, "ci_high": high, "n": len(difference)})
pd.DataFrame(reader_summary).to_csv(RESULTS / "reader_summary_dev.csv", index=False)

print(f"Prompts: {len(reader_prompts)} for {len(wrong_selected)} questions; verdicts outside the three words: {(reader['verdict'] == 'OTHER').sum()}")
print("Share of each verdict by condition, with the share of questions with a gold document in the top 5")
print(shares.round(3).to_string())
print()
print(pd.DataFrame(reader_summary).round(3).to_string(index=False))
for condition in ["wrong_wrong", "wrong_keywords"]:
    with_gold = gold_wide[condition] > 0
    print(f"{condition}: wrong candidate SUPPORTED when a gold document is in the top 5: "
          f"{(wide.loc[with_gold, condition] == 'SUPPORTED').sum()} of {with_gold.sum()}; "
          f"when none is: {(wide.loc[~with_gold, condition] == 'SUPPORTED').sum()} of {(~with_gold).sum()}")

reader_dev: 388 prompts, 0 already done, 388 to run
Prompts: 388 for 97 questions; verdicts outside the three words: 0
Share of each verdict by condition, with the share of questions with a gold document in the top 5
verdict         REFUTED  SUPPORTED  UNCLEAR   n  gold_in_top5  candidate_in_top5
condition                                                                       
true_keywords     0.814      0.113    0.072  97         0.155              0.268
true_true         0.443      0.515    0.041  97         0.546              1.062
wrong_keywords    0.763      0.072    0.165  97         0.155              0.577
wrong_wrong       0.629      0.216    0.155  97         0.134              1.216

                                      comparison   verdict  base  treated  difference  ci_low  ci_high  n
H4 wrong candidate: wrong_wrong - wrong_keywords SUPPORTED 0.072    0.216       0.144   0.062    0.227 97
H4 wrong candidate: wrong_wrong - wrong_keywords   REFUTED 0.763    0.629      -0.13

## 10. Sheet for manual labels

String matches cannot say whether a document supports or refutes a candidate, and that a gold document refutes a wrong candidate is our inference, not a fact from the dataset paper. So a person reads a sample of the documents that q_wrong ranks highest and labels each one. The sample is 30 questions with a q_wrong candidate, drawn with the fixed seed, and their top 5 corpus documents, 150 rows in all. Each row shows the same 200-word snippet the reader saw.

Columns to fill: `label` is `gold` (contains the correct answer), `echo` (mentions the candidate but does not support it), `support` (seems to support the wrong candidate) or `irrelevant`; `refutes` is `yes`, `no` or `unclear` (does the text show that the candidate is wrong?); `candidate_ok` is `yes` or `no` (is the candidate really wrong?). The sheet is written to `data/` once and never overwritten, so labels already entered are kept.

In [11]:
N_LABEL_QUESTIONS = 30
LABELS_PATH = DATA / "browsecomp-plus" / "labels_q_wrong_dev.csv"

label_questions = sorted(np.random.default_rng(SEED).choice(wrong_selected, N_LABEL_QUESTIONS, replace=False).tolist())
if LABELS_PATH.exists():
    print(f"{LABELS_PATH} already exists and is kept")
else:
    label_rows = []
    for i in label_questions:
        row = dev_rows[i]
        gold = {doc["docid"] for doc in row["gold_docs"]}
        for rank, docid in enumerate(corpus_top["wrong"][i][:READER_K], start=1):
            label_rows.append({
                "query_id": row["query_id"], "rank": rank, "docid": docid, "is_gold": docid in gold,
                "mentions_candidate": mentions(wrong_candidates[i], top_text[docid]),
                "question": row["query"], "correct_answer": row["answer"], "candidate": wrong_candidates[i],
                "snippet": snippet(top_text[docid], wrong_queries[i]),
                "label": "", "refutes": "", "candidate_ok": "",
            })
    pd.DataFrame(label_rows).to_csv(LABELS_PATH, index=False)
    print(f"Wrote {LABELS_PATH}")

labels = pd.read_csv(LABELS_PATH, keep_default_na=False)
print(f"Rows: {len(labels)} from {labels['query_id'].nunique()} questions; gold by the dataset: {labels['is_gold'].sum()}; "
      f"mention the candidate: {labels['mentions_candidate'].sum()}; labeled so far: {(labels['label'] != '').sum()}")

Wrote ../data/browsecomp-plus/labels_q_wrong_dev.csv
Rows: 150 from 30 questions; gold by the dataset: 0; mention the candidate: 54; labeled so far: 0


## Summary

### What we found (2026-11-11, development questions only)

- BM25 with the full question is at chance inside the candidate pools but well above chance over the full corpus (sections 3-4).
- A keyword query (q_keywords) retrieves gold documents better than the full question, so candidate queries are compared with q_keywords (section 5).
- Adding the correct answer (q_true) raises gold recall strongly; adding a random answer (q_random) changes it little (sections 6-7).
- H1: adding a plausible wrong candidate from the model (q_wrong) barely changes gold recall. The one interval that excludes zero (corpus recall@10) is a small drop, and intervals are not corrected for multiple comparisons, so H1 is not supported on this sample. Documents that mention the candidate do rise in the top 10 (section 8).
- H2 on retrieval: the gain from a correct candidate is far larger than the loss from a wrong one (section 8).
- H4: the reader calls the wrong candidate SUPPORTED more often after reading q_wrong results than after reading q_keywords results; the interval excludes zero (section 9). The reader answers REFUTED for most correct answers when the results do not contain them, so its REFUTED verdict often means "not shown", not "shown wrong". Only a few questions have a gold document in the top 5 of q_wrong, too few to say whether the error remains when refuting evidence is present.
- The label sheet (section 10) contains no gold document, so as built it cannot check whether gold documents refute the candidate.
- The dense retriever (H3) and the test questions have not been run.